# [프로젝트: Seq2Seq으로 한국어 번역기 만들기] 코드 구현
## Step 1. 데이터 정제 및 전처리 (Preprocessing)

In [1]:
import re
import sentencepiece as spm
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence # 길이가 다른 문장들의 길이를 똑같이 맞춰주는 파이토치의 패딩 도구

# 1. 텍스트 노이즈 전처리
# 사람이 쓴 문장에는 불필요한 기호나 띄어쓰기 오류가 많습니다. 이를 기계가 읽기 좋게 청소해 주는 함수입니다.
def preprocess_sentence(sentence, is_english=False):
    sentence = sentence.lower().strip() # 모든 알파벳을 소문자로 바꾸고, 양끝의 의미 없는 공백을 지웁니다.
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence) # 단어와 구두점(?.!,)이 붙어있으면 띄어줍니다. (예: "hi." -> "hi .")
    sentence = re.sub(r'[" "]+', " ", sentence) # 띄어쓰기가 여러 개 연속으로 있으면 한 칸으로 줄여줍니다.
    
    if is_english:
        # 영어일 경우: 알파벳과 기본 구두점만 남기고 다 지웁니다.
        sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence).strip()
    else:
        # 한국어일 경우: 한글(자음/모음 포함), 알파벳, 기본 구두점만 남깁니다. (한자나 이모티콘 등 제거)
        sentence = re.sub(r"[^ㄱ-ㅎ가-힣a-zA-Z?.!,]+", " ", sentence).strip()
    return sentence

# 2. 데이터 로드 및 중복 제거
# 원본 데이터 파일을 읽어옵니다.
with open('./korean-english-park.train/korean-english-park.train.ko', 'r', encoding='utf-8') as f:
    raw_ko = f.read().splitlines()
with open('./korean-english-park.train/korean-english-park.train.en', 'r', encoding='utf-8') as f:
    raw_en = f.read().splitlines()

# zip()으로 한국어-영어 문장을 짝지은 뒤, set()을 이용해 중복된 번역 쌍을 한 번에 제거하고 다시 리스트로 만듭니다.
cleaned_corpus = list(set(zip(raw_ko, raw_en)))

# 3. SentencePiece 학습용 텍스트 파일 저장
# SentencePiece라는 '단어 쪼개기 인공지능'을 가르치기 위해서는 순수한 텍스트 파일(.txt)이 필요합니다.
# 위에서 깨끗하게 청소한(preprocess_sentence) 문장들을 새로운 파일에 적어줍니다.
with open('ko_train.txt', 'w', encoding='utf-8') as f_ko, open('en_train.txt', 'w', encoding='utf-8') as f_en:
    for ko, en in cleaned_corpus:
        f_ko.write(preprocess_sentence(ko, is_english=False) + '\n')
        f_en.write(preprocess_sentence(en, is_english=True) + '\n')

# 4. SentencePiece 모델 학습
# 인공지능에게 "이 텍스트들을 보고, 10,000개의 가장 효율적인 단어 조각(Vocab) 사전을 만들어봐!"라고 지시합니다.
VOCAB_SIZE = 10000
print("한국어 SentencePiece 학습 중...")
spm.SentencePieceTrainer.Train(
    f'--input=ko_train.txt --model_prefix=ko_spm --vocab_size={VOCAB_SIZE} '
    # 딥러닝이 문장의 구조를 알 수 있도록 4개의 특수 토큰(기호)을 0~3번 ID로 강제 배정합니다.
    f'--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 ' 
    f'--pad_piece=<pad> --bos_piece=<start> --eos_piece=<end> --unk_piece=<unk>'
)
print("영어 SentencePiece 학습 중...")
spm.SentencePieceTrainer.Train(
    f'--input=en_train.txt --model_prefix=en_spm --vocab_size={VOCAB_SIZE} '
    f'--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 '
    f'--pad_piece=<pad> --bos_piece=<start> --eos_piece=<end> --unk_piece=<unk>'
)

# 5. 토크나이저 불러오기
# 학습이 끝나면 디스크에 저장된 단어장 모델(.model)을 파이썬 메모리로 불러옵니다.
sp_ko = spm.SentencePieceProcessor()
sp_ko.Load('ko_spm.model')
sp_en = spm.SentencePieceProcessor()
sp_en.Load('en_spm.model')

# 6. 문장 인코딩 및 필터링
enc_corpus, dec_corpus = [], []
for ko, en in cleaned_corpus:
    # 사람의 글자를 컴퓨터가 이해할 수 있는 숫자(ID) 리스트로 변환합니다. (예: "안녕" -> [45, 829])
    ko_ids = sp_ko.EncodeAsIds(preprocess_sentence(ko, False))
    # 영어(디코더의 타겟) 문장 양끝에는 번역의 시작(1=<start>)과 끝(2=<end>)을 알리는 숫자를 꼭 붙여줍니다.
    en_ids = [1] + sp_en.EncodeAsIds(preprocess_sentence(en, True)) + [2]
    
    # 너무 긴 문장은 컴퓨터 메모리를 터지게 하므로, 단어 조각이 40개 이하인 문장만 합격시킵니다.
    if len(ko_ids) <= 40 and len(en_ids) <= 40:
        # 이후 Py

한국어 SentencePiece 학습 중...
영어 SentencePiece 학습 중...


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=ko_train.txt --model_prefix=ko_spm --vocab_size=10000 --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 --pad_piece=<pad> --bos_piece=<start> --eos_piece=<end> --unk_piece=<unk>
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ko_train.txt
  input_format: 
  model_prefix: ko_spm
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentence

필터링된 최종 데이터 개수: 56485
순수 PyTorch DataLoader 구축 성공!


### Step 2. Attention 기반 Seq2Seq 모델 설계

In [2]:
import torch.nn as nn
import torch.nn.functional as F

# GPU 사용이 가능하다면 cuda, 아니면 cpu를 사용하도록 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_DIM = 256   # 단어 벡터의 크기
HIDDEN_SIZE = 512    # 모델 내부 신경망의 은닉 상태(Hidden State) 크기

# 1. Encoder 클래스: 입력 문장(한국어)을 읽고 핵심 의미를 압축함
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim) # 단어 ID를 벡터로 변환
        self.gru = nn.GRU(embedding_dim, hidden_size, batch_first=True) # 문맥 파악을 위한 GRU 레이어
        
    def forward(self, x):
        x = self.embedding(x)
        output, hidden = self.gru(x) # 입력 전체에 대한 output과 마지막 시점의 hidden state 반환
        return output, hidden

# 2. BahdanauAttention 클래스: 번역할 때 입력 문장의 어느 부분에 집중할지 결정
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.W1 = nn.Linear(hidden_size, hidden_size) # 인코더 결과값 변환용 레이어
        self.W2 = nn.Linear(hidden_size, hidden_size) # 디코더 은닉 상태 변환용 레이어
        self.V = nn.Linear(hidden_size, 1) # 점수 계산용 레이어
        
    def forward(self, hidden, enc_output):
        hidden_with_time_axis = hidden.unsqueeze(1) # 차원 맞춤 (batch, 1, hidden)
        
        # 어텐션 스코어 계산: 어떤 단어에 얼마나 집중할지 점수 산출
        score = self.V(torch.tanh(self.W1(enc_output) + self.W2(hidden_with_time_axis)))
        
        # Softmax를 적용하여 가중치 합이 1이 되도록 함 (확률 분포화)
        attention_weights = F.softmax(score, dim=1)
        
        # 가중치가 반영된 context vector 생성 (입력 정보 요약본)
        context_vector = attention_weights * enc_output
        context_vector = torch.sum(context_vector, dim=1)
        return context_vector, attention_weights

# 3. Decoder 클래스: 어텐션을 이용해 한 단어씩 번역문(영어) 생성
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.attention = BahdanauAttention(hidden_size) # 어텐션 레이어 탑재
        # 입력은 임베딩된 단어 + 어텐션 정보를 합친 크기
        self.gru = nn.GRU(embedding_dim + hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size) # 최종 단어 예측용
        
    def forward(self, x, hidden, enc_output):
        # 지금 생성할 단어가 입력 문장의 어떤 부분을 봐야 할지 결정
        context_vector, attention_weights = self.attention(hidden.squeeze(0), enc_output)
        
        x = self.embedding(x)
        # 텍스트 정보와 어텐션 요약본을 붙여서 GRU에 입력
        x = torch.cat([context_vector.unsqueeze(1), x], dim=-1)
        
        # [핵심] 이전 hidden state를 전달하여 기억을 유지하며 단어 생성
        output, hidden = self.gru(x, hidden)
        
        output = output.view(-1, output.size(2))
        x = self.fc(output) # 어떤 단어일지 확률 계산
        return x, hidden, attention_weights

# 모델 객체 생성 후 장치(GPU/CPU)에 배치
encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE).to(device)
decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE).to(device)

## Step 3. 훈련 및 번역 (Inference)

In [3]:
import torch.optim as optim
import torch.nn as nn

# 옵티마이저(Adam)와 손실 함수(CrossEntropy) 정의
# Adam은 학습률을 자동으로 조절해주어 딥러닝에서 가장 많이 쓰이는 도구입니다.
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)
# ignore_index=0은 <pad>(0번 ID)를 오차 계산에서 제외하라는 뜻입니다. (무의미한 0으로 학습 방해 방지)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 인공지능이 번역하는 함수 (학습이 끝난 후 실제로 결과를 뽑아내는 과정)
def evaluate(sentence):
    encoder.eval() # 모델을 평가 모드(가중치 고정)로 전환
    decoder.eval()
    
    sentence = preprocess_sentence(sentence, is_english=False)
    inputs = sp_ko.EncodeAsIds(sentence) # 문장을 숫자 ID 리스트로 변환
    inputs = torch.tensor(inputs).unsqueeze(0).to(device)
    
    result_ids = []
    with torch.no_grad(): # 기울기 계산을 멈춰 메모리 절약
        enc_out, enc_hidden = encoder(inputs) # 입력 문장을 인코딩하여 요약 정보(hidden) 생성
        dec_hidden = enc_hidden
        dec_input = torch.tensor([[1]]).to(device) # 첫 단어로 <start> 토큰(ID 1) 입력
        
        for t in range(40): # 최대 40단어까지 생성
            # 어텐션을 이용해 현재 시점에 가장 집중해야 할 단어를 결정하여 다음 단어 예측
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_out)
            predicted_id = predictions.argmax(1).item() # 가장 확률이 높은 단어 선택
            
            if predicted_id == 2: # <end> 토큰(ID 2)이 나오면 번역 종료
                break
                
            result_ids.append(predicted_id)
            dec_input = torch.tensor([[predicted_id]]).to(device) # 예측한 단어를 다음 입력으로 사용
            
    return sp_en.DecodeIds(result_ids) # 숫자 ID를 다시 문장으로 복원

# 10회(Epoch) 반복 학습 시작
EPOCHS = 10 

for epoch in range(EPOCHS):
    encoder.train() # 모델을 훈련 모드(가중치 업데이트 가능)로 전환
    decoder.train()
    total_loss = 0
    
    # 데이터로더에서 배치 단위로 데이터를 가져옴
    for batch, (inp, targ) in enumerate(dataloader):
        inp, targ = inp.to(device), targ.to(device)
        loss = 0
        
        enc_output, enc_hidden = encoder(inp)
        dec_hidden = enc_hidden
        dec_input = targ[:, 0].unsqueeze(1) # 디코더의 첫 입력은 <start>
        
        valid_time_steps = 0
        for t in range(1, targ.size(1)):
            # 패딩(0)은 무시하고, 실제 문장 길이에 대해서만 학습
            if targ[:, t].sum() == 0:
                break
                
            # Teacher Forcing: 실제 타겟 데이터를 디코더에 넣어 다음 단어 예측 학습
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            loss += criterion(predictions, targ[:, t]) # 오차 누적
            dec_input = targ[:, t].unsqueeze(1) # 실제 다음 정답을 입력으로 줌
            valid_time_steps += 1
            
        if valid_time_steps > 0:
            batch_loss = (loss / valid_time_steps) # 문장 내 평균 오차 계산
            total_loss += batch_loss.item()
        
            optimizer.zero_grad() # 이전 기울기 초기화
            loss.backward() # 역전파(기울기 계산)
            
            # 그래디언트 클리핑: 기울기가 너무 커져서 nan이 되지 않도록 값을 제한
            torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
            
            optimizer.step() # 가중치 업데이트
        
    print(f'Epoch {epoch+1} Loss {total_loss/len(dataloader):.4f}')
    
    # 훈련 중간마다 번역 성능 확인
    sentences = ["오바마는 대통령이다.", "시민들은 도시 속에 산다.", "커피는 필요 없다.", "일곱 명의 사망자가 발생했다."]
    print("--- [SentencePiece 번역 결과] ---")
    for s in sentences:
        print(f"K: {s} \nE: {evaluate(s)}\n")

Epoch 1 Loss 4.8334
--- [SentencePiece 번역 결과] ---
K: 오바마는 대통령이다. 
E: obama s campaign is now .

K: 시민들은 도시 속에 산다. 
E: the man was a hospital in the town of the town of the city of the island .

K: 커피는 필요 없다. 
E: the world is expected to be able to the world .

K: 일곱 명의 사망자가 발생했다. 
E: the two men were wounded in the town of the city of the city of the city .

Epoch 2 Loss 3.8563
--- [SentencePiece 번역 결과] ---
K: 오바마는 대통령이다. 
E: obama is a former president .

K: 시민들은 도시 속에 산다. 
E: the busiest is the most famoust city of the coastal town .

K: 커피는 필요 없다. 
E: the company is the most commonly symbolic .

K: 일곱 명의 사망자가 발생했다. 
E: the three day of the year old three were killed .

Epoch 3 Loss 3.3409
--- [SentencePiece 번역 결과] ---
K: 오바마는 대통령이다. 
E: obama is a very tough president obama s campaign .

K: 시민들은 도시 속에 산다. 
E: the fire is the largest city of the city .

K: 커피는 필요 없다. 
E: it is not clearly known as a precaution .

K: 일곱 명의 사망자가 발생했다. 
E: the two wounded three crew members were wounded

---

# 프로젝트 회고록: Attention 기반 Seq2Seq 번역기 구축 및 고도화

## 1. 프로젝트 개요
* **과제 내용:** 한국어 뉴스를 영어로 번역하는 Attention 기반 Seq2Seq 모델 구축
* **핵심 목표:** 텍스트 데이터 전처리, 토큰화 기법 적용, 모델 설계 및 훈련 프로세스 이해, Loss 값의 안정적 하락 증명

---

## 2. 주요 시행착오 및 오류 해결 과정

### ① 환경 설정 및 라이브러리 호환성 문제
* **시행착오:** 주피터 노트북 및 터미널에서 한국어 형태소 분석기인 `Mecab` 및 `Konlpy` 설치 중 `subprocess-exited-with-error` 에러 발생과 함께 설치 실패.
* **원인 분석:** 최신 파이썬 환경(Python 3.12)의 엄격해진 패키지 버전 표기 규칙과 옛날 방식의 구형 Mecab 패키지 간의 버전 충돌이 원인이었음.
* **해결 과정:** 에러 로그의 `Invalid version` 단서를 바탕으로 구형 패키지 대신 최신 환경과 호환되는 `mecab-python3` 패키지를 터미널에서 새로 설치하고, 주피터 커널을 재시작하여 정상 인식시킴.
* **배운 점:** 개발 환경의 버전 호환성은 프로젝트의 첫 단추이며, 무작정 구글링 코드를 복사하기보다 에러 로그를 읽고 현재 환경에 맞는 패키지를 선택하는 것이 중요함을 깨달음.

### ② 딥러닝 학습 과정에서의 '기울기 폭발 (Loss NaN)'
* **시행착오:** 모델 학습(Step 5)을 시작하자마자 `Loss` 값이 숫자가 아닌 `nan` (Not a Number)으로 출력되며 번역 결과가 `<unk>`로 완전히 무너짐.
* **원인 분석:** 1. **디코더 논리 오류:** 디코더 내 GRU 레이어 연산 시 이전 시점의 은닉 상태(`hidden state`)를 다음 시점으로 제대로 전달하지 않아 문맥 기억이 끊김.
  2. **패딩 데이터 연산 오류:** 배치 내 짧은 문장 뒤에 붙은 무의미한 패딩(0) 데이터 영역까지 무리하게 오차를 계산하려다 '0으로 나누기'와 유사한 연산 폭발 발생.
* **해결 과정:** * 디코더 GRU 코드를 `self.gru(x, hidden)` 형태로 수정하여 문맥 전달력을 복원함.
  * 학습 루프 내에 타겟 문장의 패딩(0)이 시작되면 오차 계산을 즉시 중단하는 필터링 로직 추가.
  * `torch.nn.utils.clip_grad_norm_`를 도입하여 기울기 값의 상한선(안전벨트)을 강제로 제어함.
  * 오염된 가중치를 비우기 위해 인코더/디코더 모델 설계 셀(Step 4)을 재실행하여 백지상태로 새 출발 시킴.
* **배운 점:** 딥러닝 모델은 블랙박스가 아니라 철저한 수학적 연산으로 움직인다는 점을 체감함. 특히 `nan` 발생 시 무작정 재실행하기보다 모델 파라미터 초기화와 데이터 패딩 처리를 먼저 점검해야 한다는 디버깅 자산을 얻음.

### ③ 데이터 부족 및 사전 한계로 인한 성능 저하
* **시행착오:** `Mecab` 기반 토큰화 사용 시, 단어장 크기(10,000개) 제한에 걸려 사전에 포함되지 못한 수많은 일상 단어들이 `<unk>`(Unknown)로 치환되어 번역 품질이 떨어짐.
* **원인 분석:** 약 6만 개의 훈련 데이터 세트 내에 존재하는 희귀 단어들이 단어 기반 토큰화 방식으로는 모두 버려지는 가성비 저하 문제 발생.
* **해결 과정:** 데이터 전처리 방식을 단어 기반에서 구글의 **SentencePiece (Subword Tokenization)** 기법으로 고도화함. 문장을 더 잘게 쪼개어 모르는 단어가 나오더라도 이미 알고 있는 글자 조각(Subword)들의 조합으로 인식하게 만듦.
* **배운 점:** 데이터가 부족하거나 성능 한계에 부딪혔을 때, 모델 구조만 탓할 게 아니라 **데이터를 쪼개고 다듬는 전처리(Tokenization) 방식의 변화**가 훨씬 강력하고 현실적인 돌파구가 될 수 있음을 배움.

### ④ 완벽한 번역이 나오지 않는 현상에 대한 고찰 (성능 한계)
* **현상 관찰:** SentencePiece 고도화 및 Loss의 안정적인 하락을 확인했음에도, 최종 번역 문장이 상용 번역기처럼 100% 매끄럽게 출력되지는 않음.
* **원인 분석:** 1. **데이터 양의 차이:** 상용 AI는 수백억 개의 데이터를 학습하지만, 본 프로젝트의 6만 개 데이터로는 언어의 복잡한 뉘앙스를 모두 담아내기에 물리적으로 부족함.
  2. **모델 뼈대의 한계:** Seq2Seq 아키텍처는 기계 번역의 훌륭한 뼈대이나, 길고 복잡한 문장의 문맥을 완벽히 잡기에는 태생적 한계가 존재함.
  3. **학습 시간(Epoch):** 10 Epoch라는 짧은 훈련으로는 모델이 영어 문법의 형태를 어렴풋이 흉내 내는 단계에 그칠 수밖에 없음.
* **배운 점:** 딥러닝 프로젝트의 진짜 성공 기준은 '결과물의 완벽함'에만 있는 것이 아니라, **전처리부터 모델 설계, 훈련, 에러 디버깅까지의 전체 파이프라인을 온전히 통제하고 구동시켰다는 점**에 있음을 깨달음.

---

## 3. 프로젝트 총평 및 인사이트
> "이번 프로젝트는 단순히 인공지능 코드를 돌려보는 것을 넘어, **문제를 정의하고 원인을 분석하여 제어하는 엔지니어링의 전 과정**을 경험한 값진 시간이었습니다. 
> 
> 특히 `nan` 폭발을 해결하기 위해 모델 내부의 흐름을 추적하고, 데이터의 한계를 극복하기 위해 SentencePiece 기법을 직접 이식해 보며 데이터 전처리가 모델에 미치는 영향을 깊이 깨달았습니다. 또한, 비록 100% 완벽한 번역이 나오지는 않았지만 그 원인을 '데이터 량, 학습 시간, 아키텍처의 한계'라는 공학적 시각으로 객관적으로 분석할 수 있는 시야를 갖추게 되었습니다. 
> 
> 이번에 다진 Seq2Seq와 Attention 메커니즘에 대한 탄탄한 이해를 바탕으로, 대망의 다음 아키텍처인 **트랜스포머(Transformer)** 구조도 더욱 깊이 있게 흡수할 수 있는 자신감이 생겼습니다."